# 让更加哇塞的Model接收各种输入

## 提出了哪些问题来着？

依然存在可以优化的点：

1、现在我们就直接拿最大值的index当作下一个token的index，那有没有一种可能就是，这个只是局部最优不是全局最优呢？或者说我就单纯看这个不爽，我不想每次都取最大值。

2、我们现在支持的输入token都是一样长的，但是实际情况里句子肯定有长有短的，这要怎么办呢？

上一节中把1解决了，现在的model的generate非常的哇塞

这一节解决问题2

In [2]:
# 现在model长这样
import torch
class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.k_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.v_proj = torch.nn.Linear(self.d_k, self.d_k)

        self.k_cache = None
        self.v_cache = None
        
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    def forward(self, x):
        res = x
        Q = self.q_proj(x)
        if self.k_cache is None and self.v_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        attention_weights = torch.softmax(attention_weights/torch.sqrt(self.d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        output = self.norm(output+res)

        res = output
        output = self.feed_forward(output)
        output = self.norm(output+res)
        
        return output

    def empty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

class MyModel(torch.nn.Module):
    def __init__(self, vocabulary_size=9, dims=16):
        super().__init__()
        self.vocabulary_size=vocabulary_size
        self.dims = dims
        
        self.vocabulary = torch.nn.Embedding(self.vocabulary_size, self.dims)

        self.layers = torch.nn.ModuleList(
            [Attention(dims=self.dims) for _ in range(2)]
        )

        self.lm_head = torch.nn.Linear(self.dims, self.vocabulary_size, bias=False)
    
    def empty_all_kv_cache(self):
        for layer in self.layers:
            layer.empty_kv_cache()

    def forward(self, input_ids):
        input_features = self.vocabulary(input_ids)

        hidden_states = input_features
        for layer in self.layers:
            hidden_states = layer(hidden_states)

        logits = torch.softmax(self.lm_head(hidden_states), dim=-1)

        return logits, hidden_states
    
    def sample(self, logits, top_k, top_p, temperture):
        logits = logits[:, -1, :]/temperture

        if top_k>0:
            v, _ = torch.topk(logits, top_k)
            logits[logits<v[:, [-1]]] = -1e9

        if top_p<1:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            temp_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(temp_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()

            for index in range(logits.shape[0]):
                indices_to_remove = sorted_indices[index][sorted_indices_to_remove[index]]
                logits[index, indices_to_remove] = -1e9

        probs = torch.softmax(logits, dim=-1)
        input_ids = torch.multinomial(probs, num_samples=1)
        return input_ids

    def generate(self, input_ids, max_length=10, top_k=3, top_p=0.9, temperture=1.0):
        new_token=0
        self.empty_all_kv_cache()
        
        output_ids = [input_ids,]
        
        while new_token<max_length:
            logits, _ = self.forward(input_ids=input_ids)
            
            input_ids = self.sample(logits=logits, top_k=top_k, top_p=top_p, temperture=temperture)
            
            output_ids.append(input_ids)
            new_token+=1

        output_ids = torch.cat(output_ids, dim=-1)
        return output_ids

## 思考一下思路

我们generate，再广泛一点forward，处理的肯定是(batch_size, sequence_length, dims)这样子的tensor特征，最多sequence_length不一样。

所以不管输入的是什么样的，我们都要给整合成(batch_size, sequence_length, dims)这个样子

而(batch_size, sequence_length, dims)特征对应的input_ids是什么样的呢？(batch_size, sequence_length)

ok，现在的任务就是怎么把不同长度的输入给对齐了。

In [3]:
# 先假设俩不同长度的input_id
input_id_1 = [1, 2, 3, 4]
input_id_2 = [1, 2, 3, 4, 5, 6, 7]
input_ids = [input_id_1, input_id_2]

## Embedding层接收的是什么？

一个完整的tensor，这个tensor是不能是list，所以如果这里接受到的input_ids一定是补全好了的。

那既然补全好了，说明补的东西也是一个input_id，而且这个id肯定在embedding的词表里，不然embedding会报错。

那我们就专门设定一个token来承担这个责任，不妨就设定为0，

这是我们设定的第一个special_token，后续根据需要不同我们可以设置更多的special_token来承担更多的功能

这个token，我们就叫他pad_token。

In [4]:
# 考虑到生成我们都是为了往后面生成，所以pad_token放在前面会合理一点，这样后面文字生成的就很连贯
# 但为了保持一个灵活性，我们也保留一个右pad的途径
def pad(input_ids, pad_side='left', pad_token=0):
    max_length = -1
    for input_id in input_ids:
        max_length = max(max_length, len(input_id))
    
    for index, input_id in enumerate(input_ids):
        if pad_side == 'left':
            input_ids[index] = [pad_token]*(max_length-len(input_id)) + input_id
        elif pad_side == 'right':
            input_ids[index] =  input_id + [pad_token]*(max_length-len(input_id))
        else:
            raise ValueError
    
    return input_ids

# 测试一下
print('pad之前:')
print(input_ids)
print('pad之后:')
print(pad(input_ids=input_ids, 
          pad_side='left',
          pad_token=0))

pad之前:
[[1, 2, 3, 4], [1, 2, 3, 4, 5, 6, 7]]
pad之后:
[[0, 0, 0, 1, 2, 3, 4], [1, 2, 3, 4, 5, 6, 7]]


## pad完就OK了吗？

可以确认的是pad完代码肯定可以跑通了。

但我们pad只是为了跑通吗？跑通的前提是不影响原本生成的结果。

现在这样的pad的生成结果怎么样？

In [5]:
# 为了测试准确性，我们选择用top_k=1, top_p=1, 这样不管生成几次都是一样的结果
model = MyModel()

input_ids_origin = torch.tensor([[1, 2, 3, 4]])
input_ids_pad = torch.tensor([[0, 0, 0, 1, 2, 3, 4]])

print('先看看不pad的结果')
for _ in range(3):
    print(model.generate(input_ids_origin, max_length=10, top_k=1, top_p=1))

print('再看看pad的结果')
for _ in range(3):
    print(model.generate(input_ids_pad, max_length=10, top_k=1, top_p=1))

先看看不pad的结果
tensor([[1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
再看看pad的结果
tensor([[0, 0, 0, 1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[0, 0, 0, 1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[0, 0, 0, 1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


## pad为什么会导致结果不一样？怎么解决这个问题？

原因很简单，因为pad token也是token，他本身也在被计算注意力，所以影响后面生成的结果也是很合理的。

那怎么解决呢？

记不记得前面写top-k和top-p的时候对一些token搞了个-inf，后面softmax的时候就不参考这个token了，所以如果想忽略掉这些token，把这些token也给-inf了就行了。

我们把这个过程叫做掩码，又或者叫mask。

那在sample里改o不ok？

不ok的，sample只是在最后才mask，这时候前面model的attention已经把这些pad-token给参考了。

所以如果要去掉pad-token的影响，那就要在model的forward里去把这些token给mask掉。

也就是说，我们现在要回过头改Attention类了。

In [6]:
# 尝试加一下mask在Attention类里

import torch

class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(dims, dims) # 修正了初始化，Linear输入应为int
        self.k_proj = torch.nn.Linear(dims, dims)
        self.v_proj = torch.nn.Linear(dims, dims)

        self.k_cache = None
        self.v_cache = None
        
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    # 默认mask是None，没有mask的话就是和以前一样
    def forward(self, x, attention_mask=None):
        res = x
        Q = self.q_proj(x)
        
        # KV Cache 一样不用改
        if self.k_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        # 正常计算attention_weights，shape是(batch, query_length, kv_len)
        # 在最开始的时候，shape是(batch, kv_len, kv_len), 后面就都是(batch, 1, kv_len)
        attention_weights = torch.einsum('bqd, bld->bql', Q, K)
        attention_weights = attention_weights / torch.sqrt(self.d_k)

        # mask在这里起作用
        if attention_mask is not None:
            # attention_mask 形状应该和attention_weights一样
            # padding部分的mask设置为0，将 padding 部分填充为极小负数，Softmax后权重就是0了，就不关注了
            attention_weights = attention_weights.masked_fill(attention_mask == 0, -1e9)

        attention_weights = torch.softmax(attention_weights, dim=-1)
        
        output = torch.einsum("bql, bld->bqd", attention_weights, V)
        output = self.norm(output + res)

        res = output
        output = self.feed_forward(output)
        output = self.norm(output + res)
        
        return output

    def empty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

In [7]:
# 测试一下
attention = Attention(dims=4)
attention.empty_kv_cache()

origin_input_features = torch.randn(2, 3, 4)

origin_output = attention(origin_input_features)
print(origin_output)
attention.empty_kv_cache()

# 现在在origin_input_features的前面加俩features模拟pad token的features
pad_input_features = torch.cat([torch.randn(2, 2, 4), origin_input_features], dim=1)

# 由于这个pad的length会更长，但是我们只希望看到和原来origin_output对应的那几个，所以把pad给截断
pad_output = attention(pad_input_features)[:, -origin_input_features.shape[1]:, :]
print(pad_output)

tensor([[[-1.4881,  1.1913, -0.2530,  0.5498],
         [-0.0059,  1.4025, -1.4255,  0.0289],
         [-1.2319,  1.0299, -0.7360,  0.9380]],

        [[ 0.3240, -1.7005,  0.5210,  0.8554],
         [-1.5813,  0.7277, -0.1235,  0.9770],
         [ 0.5346,  0.8721, -1.6943,  0.2876]]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[[-1.3768,  1.3291, -0.3866,  0.4343],
         [-0.0821,  1.4720, -1.3509, -0.0391],
         [-1.1872,  1.1118, -0.7844,  0.8598]],

        [[-0.1498, -1.5725,  0.9655,  0.7568],
         [-1.6339,  0.6455,  0.0330,  0.9553],
         [ 0.5456,  0.8653, -1.6949,  0.2840]]], grad_fn=<SliceBackward0>)


In [8]:
# 现在尝试一下加上mask
mask = torch.tensor([False]*(pad_input_features.shape[1]-origin_input_features.shape[1])+[True]*origin_input_features.shape[1])
attention.empty_kv_cache()
pad_output_mask = attention(pad_input_features, attention_mask=mask)[:, -origin_input_features.shape[1]:, :]
print(pad_output_mask)

tensor([[[-1.4881,  1.1913, -0.2530,  0.5498],
         [-0.0059,  1.4025, -1.4255,  0.0289],
         [-1.2319,  1.0299, -0.7360,  0.9380]],

        [[ 0.3240, -1.7005,  0.5210,  0.8554],
         [-1.5813,  0.7277, -0.1235,  0.9770],
         [ 0.5346,  0.8721, -1.6944,  0.2876]]], grad_fn=<SliceBackward0>)


In [9]:
# 看看和之前的一不一样
print(abs(origin_output-pad_output_mask)<=1e-6)

tensor([[[True, True, True, True],
         [True, True, True, True],
         [True, True, True, True]],

        [[True, True, True, True],
         [True, True, True, True],
         [True, True, True, True]]])


In [10]:
# 但好像还存在一些问题, 我们同样输入两个序列，一个长度为3，另一个截取第一个序列的前两个长度，按理说output应该是完全一致的。
attention = Attention(dims=4)
attention.empty_kv_cache()

origin_input_features = torch.randn(2, 3, 4)

origin_output = attention(origin_input_features)
print(origin_output)

part_input_features = origin_input_features[:, :2, :]
attention.empty_kv_cache()
part_output = attention(part_input_features)
print(part_output)

tensor([[[ 0.5880,  1.0991, -1.5587, -0.1284],
         [-0.3928, -1.2962,  0.2365,  1.4525],
         [ 1.1950,  0.2164, -1.5801,  0.1687]],

        [[-1.0399,  0.7157,  1.2469, -0.9227],
         [ 0.6219, -1.7235,  0.4164,  0.6852],
         [ 1.0327, -1.6106, -0.0049,  0.5827]]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[[ 0.6590,  1.0732, -1.5420, -0.1902],
         [-0.3866, -1.3066,  0.2508,  1.4424]],

        [[-0.1254,  0.6261,  1.0663, -1.5669],
         [ 1.3188, -1.1503,  0.5953, -0.7637]]],
       grad_fn=<NativeLayerNormBackward0>)


## 发现了一个问题

因为在empty_cache后的第一次输入中，attention_weights的shape也是(batch, kv_len, kv_len)

虽然我们generate时只会用到最后一个token位置上的feature，

但我们想一想，对于其他token的计算，我们现在的这种做法ok吗？

其实是不的

举个例子，对于第二个位置（不是最后一个位置）的token来说，按理说我们希望他预测的是第三个位置的token

但是由于我们没有加设定，这个时候的attention使得第三个、第四个等等后面所有的token都会被这个第二个位置的token计算到

但是他其实只需要计算第一个token的，

因此我们还需要再加一个attention_mask，由于这个mask的功能是只能看到产生他的token的信息而不能看到由他产生的token的信息

我们叫这个mask casual_mask

In [11]:
# 尝试加一下mask在Attention类里

import torch

class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(dims, dims) # 修正了初始化，Linear输入应为int
        self.k_proj = torch.nn.Linear(dims, dims)
        self.v_proj = torch.nn.Linear(dims, dims)

        self.k_cache = None
        self.v_cache = None
        
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    # 默认mask是None，没有mask的话就是和以前一样
    def forward(self, x, attention_mask=None):
        res = x
        Q = self.q_proj(x)

        if self.k_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        # 为了在第一次生成和后续生成时统一，在这里设计一下
        # 用一个三角矩阵就可以实现这个功能了，shape就是()
        q_len = Q.shape[1]
        kv_len = K.shape[1]
        causal_mask = torch.ones((q_len, kv_len), dtype=torch.bool).tril(diagonal=kv_len - q_len).unsqueeze(0)

        attention_weights = torch.einsum('bqd, bld->bql', Q, K)
        attention_weights = attention_weights / torch.sqrt(self.d_k)

        # 把俩mask合并一下
        if attention_mask is not None:
            mask = causal_mask & attention_mask
        else:
            mask = causal_mask
        
        attention_weights = attention_weights.masked_fill(mask == False, -1e9)
        
        attention_weights = torch.softmax(attention_weights, dim=-1)
        
        output = torch.einsum("bql, bld->bqd", attention_weights, V)
        output = self.norm(output + res)

        res = output
        output = self.feed_forward(output)
        output = self.norm(output + res)
        
        return output

    def empty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

In [12]:
# 测试一下
attention = Attention(dims=4)
attention.empty_kv_cache()

origin_input_features = torch.randn(2, 3, 4)

origin_output = attention(origin_input_features)
print(origin_output)

part_input_features = origin_input_features[:, :2, :]
attention.empty_kv_cache()
part_output = attention(part_input_features)
print(part_output)

print(abs(origin_output[:, :2, :]-part_output)<=1e-6)

tensor([[[ 0.2182, -1.3357,  1.4373, -0.3198],
         [-1.1183, -0.4828,  0.0149,  1.5862],
         [-0.7482,  0.0834,  1.5997, -0.9349]],

        [[-1.7052,  0.7755,  0.2933,  0.6363],
         [-1.3992,  1.4239, -0.0973,  0.0726],
         [-0.5772,  1.0396,  0.8822, -1.3445]]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[[ 0.2182, -1.3357,  1.4373, -0.3198],
         [-1.1183, -0.4828,  0.0149,  1.5862]],

        [[-1.7052,  0.7755,  0.2933,  0.6363],
         [-1.3992,  1.4239, -0.0973,  0.0726]]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[[True, True, True, True],
         [True, True, True, True]],

        [[True, True, True, True],
         [True, True, True, True]]])


In [13]:
# 再测试一下原功能
attention = Attention(dims=4)
attention.empty_kv_cache()

origin_input_features = torch.randn(2, 3, 4)

origin_output = attention(origin_input_features)
print(origin_output)
attention.empty_kv_cache()

# 现在在origin_input_features的前面加俩features模拟pad token的features
pad_input_features = torch.cat([torch.randn(2, 2, 4), origin_input_features], dim=1)

# 由于这个pad的length会更长，但是我们只希望看到和原来origin_output对应的那几个，所以把pad给截断
pad_output = attention(pad_input_features)[:, -origin_input_features.shape[1]:, :]
print(pad_output)

mask = torch.tensor([False]*(pad_input_features.shape[1]-origin_input_features.shape[1])+[True]*origin_input_features.shape[1])
attention.empty_kv_cache()
pad_output_mask = attention(pad_input_features, attention_mask=mask)[:, -origin_input_features.shape[1]:, :]
print(pad_output_mask)
# 看看和之前的一不一样
print(abs(origin_output-pad_output_mask)<=1e-6)

tensor([[[ 0.2904,  0.5130,  0.8886, -1.6920],
         [-0.1897,  1.0347,  0.7037, -1.5486],
         [ 1.4319, -1.3921,  0.0539, -0.0937]],

        [[-0.3182, -0.7374,  1.7106, -0.6549],
         [ 1.6489, -0.9895, -0.5353, -0.1242],
         [-0.2853,  0.4712,  1.2634, -1.4493]]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[[-0.1214,  1.1640,  0.5002, -1.5428],
         [-0.4324,  1.3880,  0.3677, -1.3233],
         [ 1.5048, -1.3102, -0.0974, -0.0972]],

        [[-0.4125, -0.4526,  1.7075, -0.8423],
         [ 1.5720, -1.1727, -0.0077, -0.3916],
         [-0.0032,  0.0832,  1.3724, -1.4524]]], grad_fn=<SliceBackward0>)
tensor([[[ 0.2904,  0.5130,  0.8886, -1.6920],
         [-0.1897,  1.0347,  0.7037, -1.5486],
         [ 1.4319, -1.3921,  0.0539, -0.0937]],

        [[-0.3182, -0.7374,  1.7106, -0.6549],
         [ 1.6489, -0.9895, -0.5353, -0.1242],
         [-0.2853,  0.4712,  1.2634, -1.4493]]], grad_fn=<SliceBackward0>)
tensor([[[True, True, True, True],
         [Tr

In [14]:
# ok现在只需要改generate就行了，把mask给传进去，并且每次新输出一个token懂动态改一下mask的长度
class MyModel(torch.nn.Module):
    def __init__(self, vocabulary_size=9, dims=16):
        super().__init__()
        self.vocabulary_size=vocabulary_size
        self.dims = dims
        
        self.vocabulary = torch.nn.Embedding(self.vocabulary_size, self.dims)

        self.layers = torch.nn.ModuleList(
            [Attention(dims=self.dims) for _ in range(2)]
        )

        self.lm_head = torch.nn.Linear(self.dims, self.vocabulary_size, bias=False)
    
    def empty_all_kv_cache(self):
        for layer in self.layers:
            layer.empty_kv_cache()

    # 这里改一下
    def forward(self, input_ids, attention_mask):
        input_features = self.vocabulary(input_ids)

        hidden_states = input_features
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)

        logits = torch.softmax(self.lm_head(hidden_states), dim=-1)

        return logits, hidden_states
    
    def sample(self, logits, top_k, top_p, temperture):
        logits = logits[:, -1, :]/temperture

        if top_k>0:
            v, _ = torch.topk(logits, top_k)
            logits[logits<v[:, [-1]]] = -1e9

        if top_p<1:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            temp_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(temp_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()

            for index in range(logits.shape[0]):
                indices_to_remove = sorted_indices[index][sorted_indices_to_remove[index]]
                logits[index, indices_to_remove] = -1e9

        probs = torch.softmax(logits, dim=-1)
        input_ids = torch.multinomial(probs, num_samples=1)
        return input_ids

    def generate(self, input_ids, max_length=10, top_k=3, top_p=0.9, temperture=1.0):
        new_token=0
        self.empty_all_kv_cache()
        
        output_ids = [input_ids,]

        # 搞起来搞起来
        pad_token = 0
        attention_mask = input_ids!=pad_token

        while new_token<max_length:
            logits, _ = self.forward(input_ids=input_ids, attention_mask=attention_mask)
            
            input_ids = self.sample(logits=logits, top_k=top_k, top_p=top_p, temperture=temperture)
            
            output_ids.append(input_ids)
            new_token+=1

            # 新token生成以后也搞后面去
            new_mask = torch.ones((input_ids.shape[0], 1), dtype=torch.bool)
            attention_mask = torch.cat([attention_mask, new_mask], dim=1)

        output_ids = torch.cat(output_ids, dim=-1)
        return output_ids

In [15]:
# 为了测试准确性，我们还选择用top_k=1, top_p=1
input_ids_origin = torch.tensor([[1, 2, 3, 4]])
input_ids_pad = torch.tensor([[0, 0, 0, 1, 2, 3, 4]])

print('先看看不pad的结果')
for _ in range(3):
    print(model.generate(input_ids_origin, max_length=10, top_k=1, top_p=1))

print('再看看pad的结果')
for _ in range(3):
    print(model.generate(input_ids_pad, max_length=10, top_k=1, top_p=1))

先看看不pad的结果
tensor([[1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
再看看pad的结果
tensor([[0, 0, 0, 1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[0, 0, 0, 1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[0, 0, 0, 1, 2, 3, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


## 至此，第一节结束

恭喜你搭建出来了一个非常哇塞的可以逐个吐token的Model，

也是当前所有大模型产品的基本范式。